# CDCN CVPR 2020 · Lượt chạy thử Train/Dev trên OULU-NPU
Chọn **GPU A100** và chạy các cell theo thứ tự. Lượt này dùng 3 video Train và
3 video Dev, mỗi video có 8 khung. Chạy 3 epoch từ khởi tạo mới. Mỗi epoch lấy
ngẫu nhiên một khung mỗi video Train. Dev dùng đủ 8 khung và tính trung bình
các tỷ số tổng depth dự đoán trên tổng map PRNet.

Cell 3 khôi phục ZIP tiền xử lý trên Drive nếu dấu vân tay cấu hình trùng.
Nếu cần dựng từ đầu, cell đó đọc các video được chọn trong TAR và ghi
snapshot lên Drive. Cell 4 lưu checkpoint, điểm video, chỉ số Dev và thời
gian. Chỉ số của 3 video Dev dùng để kiểm tra pipeline. Mốc của paper dùng
toàn bộ Protocol 1 và chấm trên Test.

Nếu đã có kết quả pilot, chạy cell 1–3 rồi chuyển thẳng đến cell 5 để đo
batch trên A100 bằng các khung Train đã có.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-train-pilot")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-train-pilot-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn runtime GPU trước khi tiếp tục")
print("GPU:", torch.cuda.get_device_name(), "| PyTorch:", torch.__version__)

In [ ]:
#@title 2. Nạp code đi kèm
BUNDLE_SHA256 = '7167a15afe595354e5c4e02542dd7f95adea40ab310473f3cac039866eee4c51'
BUNDLE = (
    '...'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})


In [ ]:
#@title 3. Khôi phục hoặc dựng 48 khung MTCNN–PRNet
def prepare_prnet_environment():
    global PY, PRNET, RENDERER
    run("install-uv", sys.executable, "-m", "pip", "install", "uv")
    VENV = WORK / "venv"
    PY = VENV / "bin/python"
    if not PY.is_file():
        run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
    run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
        *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))
    PRNET = WORK / "PRNet"
    RENDERER = WORK / "PRNet-Depth-Generation"
    for path, repo, commit in [
        (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
        (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
    ]:
        if not path.exists():
            run(f"clone-{path.name}", "git", "clone", repo, path)
        run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
    WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
    def weight_matches(path):
        if not path.is_file():
            return False
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

    if not weight_matches(WEIGHT):
        partial = WEIGHT.with_suffix(".part")
        run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
            "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
        if not weight_matches(partial):
            raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
        partial.replace(WEIGHT)

import json
import zipfile

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

archives = {}
for name in ("Train_files.tar", "Dev_files.tar"):
    path = DATA / name
    with path.open("rb") as stream:
        archives[name] = {"bytes": path.stat().st_size,
                          "prefix_sha256": hashlib.sha256(stream.read(65536)).hexdigest()}
worker = (WORK / "scripts/cdcn_prnet_smoke.py").read_text(encoding="utf-8")
key = {"profile": "p1-3-videos-per-split-8-frames",
       "protocol": sha(DATA / "Protocols.tar"), "archives": archives,
       "preparer": hashlib.sha256(worker.encode("utf-8")).hexdigest()}
key_hash = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:20]
cache = RUN.parent / "prepared-cache" / key_hash
cache.mkdir(parents=True, exist_ok=True)
assets = (WORK / f"assets-{time.time_ns()}").resolve()
cache_meta = cache / "snapshot.json"
if cache_meta.is_file():
    with Progress("Restore RGB/bbox/PRNet from Drive"):
        meta = json.loads(cache_meta.read_text())
        digest = meta["sha256"]
        if meta["key"] != key or len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest):
            raise ValueError(f"Invalid snapshot metadata: {cache_meta}")
        snapshot = cache / f"assets-{digest}.zip"
        local = assets.with_suffix(".zip")
        shutil.copyfile(snapshot, local)
        if digest != sha(local):
            raise ValueError(f"Snapshot verification failed: {snapshot}")
        with zipfile.ZipFile(local) as archive:
            if any(not (assets / name).resolve().is_relative_to(assets) for name in archive.namelist()):
                raise ValueError("Invalid archive path")
            archive.extractall(assets)
        local.unlink()
else:
    prepare_prnet_environment()
    run("prepare-48-frames", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
        "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--training-pilot", "--output", assets)
    with Progress("Save RGB/bbox/PRNet to Drive"):
        local = Path(shutil.make_archive(str(assets), "zip", assets))
        digest = sha(local)
        snapshot = cache / f"assets-{digest}.zip"
        partial = cache / f"{RUN.name}-assets.zip.part"
        shutil.copyfile(local, partial)
        if sha(partial) != digest:
            raise ValueError("Snapshot copy differs from local ZIP")
        partial.replace(snapshot)
        metadata = cache / f"{RUN.name}-snapshot.json.part"
        metadata.write_text(json.dumps({"key": key, "sha256": digest}, indent=2))
        metadata.replace(cache_meta)
        local.unlink()
shutil.copyfile(assets / "smoke_summary.json", RUN / "preparation.json")
print("PILOT_DATA_READY", snapshot)

In [ ]:
#@title 4. Train CDCN, đọc lại checkpoint và tính điểm Dev
EPOCHS = 3
BATCH = 3
SEED = 43
result = RUN / f"train-{time.time_ns()}"
run("train-and-dev", sys.executable, "-u", WORK / "scripts/cdcn_prnet_train_pilot.py",
    "--assets", assets, "--output", result, "--epochs", EPOCHS, "--batch", BATCH,
    "--seed", SEED, "--device", "cuda")
print((result / "pilot_result.json").read_text())
print("RUN_DIR", result)

In [ ]:
#@title 5. Đo batch 8, 16, 32 và 56 trên A100
PROBE_BATCHES = (8, 16, 32, 56)
report = RUN / f"batch-probe-{time.time_ns()}.json"
run("batch-probe", sys.executable, "-u", WORK / "scripts/cdcn_prnet_batch_probe.py",
    "--assets", assets, "--output", report, "--warmup", 2, "--steps", 6,
    "--batches", *PROBE_BATCHES)
print(report.read_text())